# Phase 2: AI Enrichment & Entity Extraction Pipeline

## 1. NLP Logic: Mapping Unstructured Text to Player Entities
* Leveraging LLMs to bridge the gap between unstructured social media text and relational database records.

In [1]:
import pandas as pd
import json
from openai import OpenAI

print("Initializing AI Sentiment Engine...")

# 1. The Mock Data (Simulating what Reddit will eventually give us)
mock_reddit_data = {
    'Comment_ID': ['r1', 'r2', 'r3', 'r4', 'r5'],
    'Text': [
        "Sabrina's three-point shooting was absolutely unreal tonight! MVP!",
        "I'm so frustrated with the coaching down the stretch, why didn't we call a timeout?",
        "Just bought the new seafoam Liberty jersey, shipping took forever though.",
        "Stewie's defense in the paint saved the game.",
        "Tickets at Barclays are getting way too expensive this season."
    ]
}

# Load into a Pandas DataFrame
df_comments = pd.DataFrame(mock_reddit_data)
print(f"Loaded {len(df_comments)} mock comments for processing.")

# 2. Setup the AI Client (You will need to replace 'YOUR_API_KEY' with a real key later)
# client = OpenAI(api_key="YOUR_API_KEY")

# 3. Define the AI Prompt (The "Instructions" for the AI)
def analyze_comment_with_ai(text):
    """
    Sends text to the AI and forces it to return a structured JSON response.
    """
    prompt = f"""
    You are an expert sports data analyst. Read the following fan comment and categorize it.
    Return ONLY a valid JSON object with exactly two keys: "Sentiment" and "Topic".
    
    Rules:
    - "Sentiment" must be exactly one of: 'Positive', 'Negative', 'Neutral'.
    - "Topic" must be exactly one of: 'Player Performance', 'Coaching', 'Merchandise', 'Ticketing/Arena', 'General'.
    
    Comment: "{text}"
    """
    
    # --- NOTE: This block is commented out until you drop in a real API key! ---
    # response = client.chat.completions.create(
    #     model="gpt-3.5-turbo",
    #     messages=[{"role": "user", "content": prompt}],
    #     temperature=0.0 
    # )
    # return json.loads(response.choices[0].message.content)
    
    # --- Mocking the AI response for testing purposes ---
    if "MVP" in text or "saved the game" in text:
        return {"Sentiment": "Positive", "Topic": "Player Performance"}
    elif "coaching" in text:
        return {"Sentiment": "Negative", "Topic": "Coaching"}
    elif "jersey" in text:
        return {"Sentiment": "Neutral", "Topic": "Merchandise"}
    else:
        return {"Sentiment": "Negative", "Topic": "Ticketing/Arena"}

# 4. The Processing Loop
print("Sending data to AI for processing...")

results = []
for index, row in df_comments.iterrows():
    # Pass each comment to the AI
    ai_output = analyze_comment_with_ai(row['Text'])
    
    # Combine the original data with the new AI insights
    results.append({
        'Comment_ID': row['Comment_ID'],
        'Text': row['Text'],
        'Sentiment': ai_output['Sentiment'],
        'Topic': ai_output['Topic']
    })

# 5. The Final Output
df_ai_results = pd.DataFrame(results)

print("\n--- AI Extraction Complete ---")
print(df_ai_results.to_string(index=False))

Initializing AI Sentiment Engine...
Loaded 5 mock comments for processing.
Sending data to AI for processing...

--- AI Extraction Complete ---
Comment_ID                                                                                Text Sentiment              Topic
        r1                  Sabrina's three-point shooting was absolutely unreal tonight! MVP!  Positive Player Performance
        r2 I'm so frustrated with the coaching down the stretch, why didn't we call a timeout?  Negative           Coaching
        r3           Just bought the new seafoam Liberty jersey, shipping took forever though.   Neutral        Merchandise
        r4                                       Stewie's defense in the paint saved the game.  Positive Player Performance
        r5                      Tickets at Barclays are getting way too expensive this season.  Negative    Ticketing/Arena


In [2]:
from sqlalchemy import create_engine

# 2. Define your SQL Server details

In [3]:
server_name = 'Your_Server_Name' # This depends on your setup!
database_name = 'LibertyAnalytics'

# 3. Create the connection string

In [4]:
connection_string = f"mssql+pyodbc://@{server_name}/{database_name}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = create_engine(connection_string)

# 4. Create the Table

In [5]:
from sqlalchemy import text

print("Building Fact_Fan_Sentiment table in SQL Server...")

create_sentiment_table = """
    -- Drop the table if it already exists so we can rerun this cleanly
    IF OBJECT_ID('Fact_Fan_Sentiment', 'U') IS NOT NULL
        DROP TABLE Fact_Fan_Sentiment;

    CREATE TABLE Fact_Fan_Sentiment (
        Sentiment_ID INT IDENTITY(1,1) PRIMARY KEY,  -- Auto-incrementing ID
        Comment_ID VARCHAR(50) NOT NULL,             -- Reddit's unique ID for the comment
        TEAM_ID INT,                                 -- Foreign Key connecting to Dim_Teams
        Comment_Date DATE,                           -- Foreign Key connecting to Dim_Dates
        Raw_Text NVARCHAR(MAX),                      -- The actual text of the comment
        Sentiment VARCHAR(50),                       -- The AI output (Positive/Negative/Neutral)
        Topic VARCHAR(50)                            -- The AI output (Coaching/Merch/etc.)
    );
"""

# Execute the SQL command using your existing engine
with engine.connect() as conn:
    conn.execute(text(create_sentiment_table))
    conn.commit()

print("SUCCESS! Fact_Fan_Sentiment has been added to your Star Schema.")

Building Fact_Fan_Sentiment table in SQL Server...
SUCCESS! Fact_Fan_Sentiment has been added to your Star Schema.


## 5. Load the Data

In [6]:
print("Transforming and Loading AI Sentiment data...")

# 1. Add our Foreign Keys so it connects to the Star Schema
# (We are hardcoding a NY Liberty Team ID and a Game Date for this mock data)
df_ai_results['TEAM_ID'] = 1611661313  # Replace '1' with the actual TEAM_ID for NYL in your database
df_ai_results['Comment_Date'] = '2021-05-14' # A sample game date from your Dim_Dates

# 2. Rename the column to match the SQL table perfectly
df_to_load = df_ai_results.rename(columns={'Text': 'Raw_Text'})

# 3. Load it into SQL Server!
df_to_load.to_sql(
    name='Fact_Fan_Sentiment',
    con=engine,
    if_exists='append', 
    index=False         
)

print("SUCCESS! AI Pipeline complete. Fan sentiment is now in the Data Warehouse.")

Transforming and Loading AI Sentiment data...
SUCCESS! AI Pipeline complete. Fan sentiment is now in the Data Warehouse.


## 2. Schema Evolution: Integrating AI Insights into the Warehouse
* Evolving the Fact_Fan_Sentiment table to support Player-Level grain, allowing for 'Mixed Grain' joins.

In [7]:
from sqlalchemy import text

print("Altering Fact_Fan_Sentiment to support Player-Level Grain...")

# The SQL command to add the new column
alter_table_query = """
    ALTER TABLE Fact_Fan_Sentiment
    ADD PLAYER_ID INT NULL;
"""

# Execute the alter command
with engine.connect() as conn:
    conn.execute(text(alter_table_query))
    conn.commit()

print("Success! The table is now ready for Mixed Grain data.")

Altering Fact_Fan_Sentiment to support Player-Level Grain...
Success! The table is now ready for Mixed Grain data.


In [8]:
def analyze_comment_with_ai_v2(text):
    """
    Smarter AI that extracts Sentiment, Topic, AND Player Mentions.
    """
    prompt = f"""
    You are a sports data analyst. Analyze this WNBA fan comment.
    Return ONLY a valid JSON object with these three keys: "Sentiment", "Topic", "Player_Mentioned".
    
    Rules:
    - "Sentiment": 'Positive', 'Negative', or 'Neutral'.
    - "Topic": 'Player Performance', 'Coaching', 'Merchandise', 'Ticketing/Arena', or 'General'.
    - "Player_Mentioned": If a specific player is mentioned, return their Full Name. 
      If it's a general team comment, return null.
    
    Comment: "{text}"
    """
    
    # (Mocking the AI logic for your test run)
    if "Sabrina" in text:
        return {"Sentiment": "Positive", "Topic": "Player Performance", "Player_Mentioned": "Sabrina Ionescu"}
    elif "Laney" in text or "Betnijah" in text:
        return {"Sentiment": "Positive", "Topic": "Player Performance", "Player_Mentioned": "Betnijah Laney-Hamilton"}
    else:
        return {"Sentiment": "Negative", "Topic": "Coaching", "Player_Mentioned": None}

In [9]:
# 4. The Processing Loop
print("Sending data to AI for processing...")

results = []
for index, row in df_comments.iterrows():
    # Pass each comment to the AI
    ai_output = analyze_comment_with_ai_v2(row['Text'])
    
    # Combine the original data with the new AI insights
    results.append({
        'Comment_ID': row['Comment_ID'],
        'Text': row['Text'],
        'Sentiment': ai_output['Sentiment'],
        'Topic': ai_output['Topic'],
        'Player_Mentioned': ai_output['Player_Mentioned']
    })

# 5. The Final Output
df_ai_results = pd.DataFrame(results)

print("\n--- AI Extraction Complete ---")
print(df_ai_results.to_string(index=False))

Sending data to AI for processing...

--- AI Extraction Complete ---
Comment_ID                                                                                Text Sentiment              Topic Player_Mentioned
        r1                  Sabrina's three-point shooting was absolutely unreal tonight! MVP!  Positive Player Performance  Sabrina Ionescu
        r2 I'm so frustrated with the coaching down the stretch, why didn't we call a timeout?  Negative           Coaching             None
        r3           Just bought the new seafoam Liberty jersey, shipping took forever though.  Negative           Coaching             None
        r4                                       Stewie's defense in the paint saved the game.  Negative           Coaching             None
        r5                      Tickets at Barclays are getting way too expensive this season.  Negative           Coaching             None


In [10]:
from sqlalchemy import text

# Let's find Betnijah's ACTUAL Player ID from your Dim_Players table
find_id_query = "SELECT PLAYER_ID FROM Dim_Players WHERE PLAYER_NAME LIKE '%Laney%'"

with engine.connect() as conn:
    real_p_id = conn.execute(text(find_id_query)).fetchone()[0]
    
    # Now, update your sentiment table so the mock comments actually point to her
    update_query = f"UPDATE Fact_Fan_Sentiment SET PLAYER_ID = {real_p_id} WHERE TEAM_ID = 1611661313"
    conn.execute(text(update_query))
    conn.commit()

print(f"Fixed! Betnijah's ID ({real_p_id}) is now linked to your sentiment data.")

Fixed! Betnijah's ID (204335) is now linked to your sentiment data.


In [11]:
def analyze_comment_with_ai_v2(text):
    # Setup a base dictionary
    result = {"Sentiment": "Neutral", "Topic": "General", "Player_Mentioned": None}
    
    # Check each condition independently (No elif = No indentation confusion)
    if "Sabrina" in text:
        result = {"Sentiment": "Positive", "Topic": "Player Performance", "Player_Mentioned": "Sabrina Ionescu"}
    
    if "Laney" in text or "Betnijah" in text:
        result = {"Sentiment": "Positive", "Topic": "Player Performance", "Player_Mentioned": "Betnijah Laney-Hamilton"}
        
    if "Stewie" in text or "defense" in text:
        result = {"Sentiment": "Positive", "Topic": "Player Performance", "Player_Mentioned": "Breanna Stewart"}
        
    return result

In [12]:
# 4. The Processing Loop
print("Sending data to AI for processing...")

results = []
for index, row in df_comments.iterrows():
    # Pass each comment to the AI
    ai_output = analyze_comment_with_ai_v2(row['Text'])
    
    # Combine the original data with the new AI insights
    results.append({
        'Comment_ID': row['Comment_ID'],
        'Text': row['Text'],
        'Sentiment': ai_output['Sentiment'],
        'Topic': ai_output['Topic'],
        'Player_Mentioned': ai_output['Player_Mentioned']
    })

# 5. The Final Output
df_ai_results = pd.DataFrame(results)

print("\n--- AI Extraction Complete ---")
print(df_ai_results.to_string(index=False))

Sending data to AI for processing...

--- AI Extraction Complete ---
Comment_ID                                                                                Text Sentiment              Topic Player_Mentioned
        r1                  Sabrina's three-point shooting was absolutely unreal tonight! MVP!  Positive Player Performance  Sabrina Ionescu
        r2 I'm so frustrated with the coaching down the stretch, why didn't we call a timeout?   Neutral            General             None
        r3           Just bought the new seafoam Liberty jersey, shipping took forever though.   Neutral            General             None
        r4                                       Stewie's defense in the paint saved the game.  Positive Player Performance  Breanna Stewart
        r5                      Tickets at Barclays are getting way too expensive this season.   Neutral            General             None


In [13]:
from sqlalchemy import text

# 1. Get the actual IDs for our three stars
with engine.connect() as conn:
    stewie_id = conn.execute(text("SELECT PLAYER_ID FROM Dim_Players WHERE PLAYER_NAME LIKE '%Stewart%'")).fetchone()[0]
    sabrina_id = conn.execute(text("SELECT PLAYER_ID FROM Dim_Players WHERE PLAYER_NAME LIKE '%Ionescu%'")).fetchone()[0]
    laney_id = conn.execute(text("SELECT PLAYER_ID FROM Dim_Players WHERE PLAYER_NAME LIKE '%Laney%'")).fetchone()[0]

    # 2. Update specific comments to match the right players
    # Link 'r4' (Stewie's defense) to Breanna Stewart
    conn.execute(text(f"UPDATE Fact_Fan_Sentiment SET PLAYER_ID = {stewie_id} WHERE Comment_ID = 'r4'"))
    
    # Link 'r1' (Sabrina's shooting) to Sabrina Ionescu
    conn.execute(text(f"UPDATE Fact_Fan_Sentiment SET PLAYER_ID = {sabrina_id} WHERE Comment_ID = 'r1'"))
    
    # Ensure other comments stay with Betnijah (or set to NULL if general)
    conn.execute(text(f"UPDATE Fact_Fan_Sentiment SET PLAYER_ID = {laney_id} WHERE Comment_ID IN ('r2', 'r3', 'r5')"))
    
    conn.commit()

print(f"IDs Aligned: Stewie ({stewie_id}), Sabrina ({sabrina_id}), and Laney ({laney_id}) are now correctly mapped!")

IDs Aligned: Stewie (1627668), Sabrina (1629477), and Laney (204335) are now correctly mapped!


## Find a real date in the database where Breanna Stewart had a "Defensive Masterclass" for the Liberty and update the mock data to use that date.


In [14]:
from sqlalchemy import text

print("Finding a real defensive game for Stewie...")

# 1. Find the date of a high-defensive-stock game for Breanna Stewart
find_stewie_game = """
    SELECT TOP 1 fg.GAME_DATE, p.PLAYER_ID
    FROM Fact_Player_Stats fps
    JOIN Dim_Players p ON fps.PLAYER_ID = p.PLAYER_ID
    JOIN Fact_Games fg ON fps.GAME_ID = fg.GAME_ID
    JOIN Dim_Teams t ON fg.TEAM_ID = t.TEAM_ID
    WHERE p.PLAYER_NAME LIKE '%Stewart%' 
      AND t.TEAM_ABBREVIATION = 'NYL'
      AND (CAST(fps.Steals AS INT) + CAST(fps.Blocks AS INT)) >= 3
    ORDER BY fg.GAME_DATE DESC;
"""

with engine.connect() as conn:
    game_info = conn.execute(text(find_stewie_game)).fetchone()
    
    if game_info:
        real_date = game_info[0]
        stewie_id = game_info[1]
        
        print(f"Found a match! Stewie had a big defensive game on {real_date}.")
        
        # 2. Update the 'r4' comment (Stewie's defense) to match this REAL date and ID
        update_query = f"""
            UPDATE Fact_Fan_Sentiment 
            SET Comment_Date = '{real_date}', 
                PLAYER_ID = {stewie_id} 
            WHERE Comment_ID = 'r4';
        """
        conn.execute(text(update_query))
        conn.commit()
        print(f"SUCCESS: Comment 'r4' is now synced to {real_date} for Player ID {stewie_id}.")
    else:
        print("Could not find a defensive game for Stewie. Check if her data was loaded correctly.")

Finding a real defensive game for Stewie...
Found a match! Stewie had a big defensive game on 2025-09-09.
SUCCESS: Comment 'r4' is now synced to 2025-09-09 for Player ID 1627668.


## Find a real date in the database where Sabrina Ionescu was on fire from the top of the key and update the mock data to use that date.

In [15]:
from sqlalchemy import text

# Find a game where Sabrina made 4+ three-pointers
find_sabrina_game = """
    SELECT TOP 1 fg.GAME_DATE, p.PLAYER_ID, fps.Three_Pointers_Made
    FROM Fact_Player_Stats fps
    JOIN Dim_Players p ON fps.PLAYER_ID = p.PLAYER_ID
    JOIN Fact_Games fg ON fps.GAME_ID = fg.GAME_ID
    JOIN Dim_Teams t ON fg.TEAM_ID = t.TEAM_ID
    WHERE p.PLAYER_NAME LIKE '%Ionescu%' 
      AND t.TEAM_ABBREVIATION = 'NYL'
      AND CAST(fps.Three_Pointers_Made AS INT) >= 4
    ORDER BY fg.GAME_DATE DESC;
"""

with engine.connect() as conn:
    game_info = conn.execute(text(find_sabrina_game)).fetchone()
    
    if game_info:
        s_date, s_id, s_threes = game_info
        print(f"Match Found! Sabrina made {s_threes} threes on {s_date}.")
        
        # Sync the 'r1' comment (Sabrina's shooting) to this real game
        conn.execute(text(f"UPDATE Fact_Fan_Sentiment SET Comment_Date = '{s_date}', PLAYER_ID = {s_id} WHERE Comment_ID = 'r1'"))
        conn.commit()
        print(f"SUCCESS: Comment 'r1' is now synced to Sabrina's masterclass on {s_date}.")

Match Found! Sabrina made 4 threes on 2025-08-19.
SUCCESS: Comment 'r1' is now synced to Sabrina's masterclass on 2025-08-19.


In [16]:
from sqlalchemy import text

print("Aligning IDs with 'TOP 1' safety logic...")

with engine.connect() as conn:
    # 1. Start fresh: Clear out all previous player links
    conn.execute(text("UPDATE Fact_Fan_Sentiment SET PLAYER_ID = NULL"))
    
    # 2. Re-link Sabrina (using TOP 1 to prevent errors)
    conn.execute(text("""
        UPDATE Fact_Fan_Sentiment 
        SET PLAYER_ID = (SELECT TOP 1 PLAYER_ID FROM Dim_Players WHERE PLAYER_NAME LIKE '%Ionescu%') 
        WHERE Comment_ID = 'r1'
    """))
    
    # 3. Re-link Stewie (using TOP 1 to prevent errors)
    conn.execute(text("""
        UPDATE Fact_Fan_Sentiment 
        SET PLAYER_ID = (SELECT TOP 1 PLAYER_ID FROM Dim_Players WHERE PLAYER_NAME LIKE '%Stewart%') 
        WHERE Comment_ID = 'r4'
    """))
    
    # 4. Re-link Betnijah (since your table showed she scored 30, let's keep her stats clean)
    # We'll link a new mock comment or just use the date-based join later.
    # For now, we'll just link her name to her ID specifically if needed.
    
    conn.commit()

print("SUCCESS: Player IDs are now precisely mapped without subquery errors.")

Aligning IDs with 'TOP 1' safety logic...
SUCCESS: Player IDs are now precisely mapped without subquery errors.


In [17]:
from sqlalchemy import text

# Add a specific performance comment for Betnijah
new_comment = {
    'Comment_ID': 'r6',
    'Raw_Text': "Betnijah Laney is absolutely carryng this team right now. 30 points?! Legend.",
    'Sentiment': 'Positive',
    'Topic': 'Player Performance',
    'Comment_Date': '2021-05-14',
    'TEAM_ID': 1611661313
}

with engine.connect() as conn:
    # 1. Insert the new comment
    conn.execute(text("""
        INSERT INTO Fact_Fan_Sentiment (Comment_ID, Raw_Text, Sentiment, Topic, Comment_Date, TEAM_ID)
        VALUES (:Comment_ID, :Raw_Text, :Sentiment, :Topic, :Comment_Date, :TEAM_ID)
    """), new_comment)
    
    # 2. Map it to Betnijah's ID
    conn.execute(text("""
        UPDATE Fact_Fan_Sentiment 
        SET PLAYER_ID = (SELECT TOP 1 PLAYER_ID FROM Dim_Players WHERE PLAYER_NAME LIKE '%Laney%')
        WHERE Comment_ID = 'r6'
    """))
    conn.commit()

print("Betnijah now has a dedicated performance comment in the database!")

Betnijah now has a dedicated performance comment in the database!
